In [1]:
import numpy as np
import scipy.stats
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import seaborn as sns
import statsmodels.api

### IMPORT DATA


In [2]:
bigFive = ['Extraversion', 'Agreeableness', 'Conscientiousness', 'Neuroticism', 'Openness']
judges = ['Judge 1', 'Judge 2', 'Judge 3', 'Judge 4', 'Judge 5', 'Judge 6', 'Judge 7', 'Judge 8', 'Judge 9', 'Judge 10', 'Judge 11']

#Function taking the judge number 01-11 to return their table of scores
def printJudge(judgeNo):
   # [640rows x 6 columns]
    # Columns: Clip_ID (0), Extraversion (1), Agreeableness (2), Conscientiousness (3), Neuroticism (4), Openness (5)
   return pd.read_csv(f'data/Personality_Scores/Score_0{judgeNo}.csv', delimiter=',')

# [640rows x 4 columns]
#  Columns: Clip_ID (0), Speaker_ID (1), Gender (2), Status (3)
metadata = pd.read_csv('data/metadata.csv')

# Data per Judge
judge1= np.array(printJudge('01'))
judge2= np.array(printJudge('02'))
judge3= np.array(printJudge('03'))
judge4= np.array(printJudge('04'))
judge5= np.array(printJudge('05'))
judge6= np.array(printJudge('06'))
judge7= np.array(printJudge('07'))
judge8= np.array(printJudge('08'))
judge9= np.array(printJudge('09'))
judge10= np.array(printJudge('10'))
judge11= np.array(printJudge('11'))



### DATA MANIPULATION

In [3]:
# ALL INFO FOR A JUDGE FOR ONE GENDER
# Connect Data  parameters --> 'F' or 'M' , judge1 etc.
def splitPerGender(gender, judgeNo):
    judgeData = judgeNo
    listOfGender = np.where(np.array(metadata)[:,2] == gender)[0]
    judgeGenderData = judgeData[listOfGender]
    return (judgeGenderData)


##### Ways to average the scores

In [4]:
#MEAN Scores per clip between judges - 640x5 scores 
def meanScoresPerClip():
    clipIDs = judge1[:,0]
    result = []
    allScores=[]
    for i in range(11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        scores = np.array(np.array(printJudge(x))[:,1:])
        allScores.append(scores)
    allScores = np.stack(allScores, axis=0)
    meanScoresPerClip = np.mean(allScores, axis=0)
    result = np.column_stack((np.array(clipIDs), meanScoresPerClip))
    return result


In [5]:
#MEDIAN Scores per clip between judges - 640x5 scores
def medianScoresPerClip():
    clipIDs = judge1[:,0]
    result = []
    allScores=[]
    for i in range(11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        scores = np.array(np.array(printJudge(x))[:,1:])
        allScores.append(scores)
    allScores = np.stack(allScores, axis=0)
    medianScoresPerClip = np.median(allScores, axis=0)
    medianScoresPerClip = medianScoresPerClip.astype(int)
    result = np.column_stack((np.array(clipIDs), medianScoresPerClip))
    return result

In [6]:
#MODE Scores per clip between judges
def calcMode(x):
    unique_elements, counts = np.unique(x, return_counts=True)
    mode_index = np.argmax(counts)
    return unique_elements[mode_index]

def modeScoresPerClip():
    clipIDs = judge1[:,0]
    result = []
    allScores=[]
    for i in range(11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        scores = np.array(np.array(printJudge(x))[:,1:])
        allScores.append(scores)
    allScores = np.stack(allScores, axis=0)
    allScores = np.transpose(allScores, (1,2,0))
    mode_result = np.apply_along_axis(calcMode, axis=2, arr=allScores)
    result = np.column_stack((np.array(clipIDs), mode_result))
    print(result)


In [7]:
#Mean Scores per clip between judges - ONE GENDER

def meanScoresPerClipGender(gender):
    listOfGender = np.where(np.array(metadata)[:,2] == gender)[0]
    clipIDsForGender = judge1[:,0][listOfGender]
    result = []
    allScores=[]
    i = 0
    while (i<11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        scores = np.array(np.array(printJudge(x))[:,1:][listOfGender])
        allScores.append(scores)
        i = i + 1
    allScores = np.stack(allScores, axis=0)
    meanScoresPerClip = np.mean(allScores, axis=0)
    result = np.column_stack((np.array(clipIDsForGender), meanScoresPerClip))
    return result

def medianScoresPerClipGender(gender):
    listOfGender = np.where(np.array(metadata)[:,2] == gender)[0]
    clipIDsForGender = judge1[:,0][listOfGender]
    result = []
    allScores=[]
    i = 0
    while (i<11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        scores = np.array(np.array(printJudge(x))[:,1:][listOfGender])
        allScores.append(scores)
        i = i + 1
    allScores = np.stack(allScores, axis=0)
    meanScoresPerClip = np.median(allScores, axis=0)
    result = np.column_stack((np.array(clipIDsForGender), meanScoresPerClip))
    return result

## Distribution of data

In [8]:
#Box plot of mean data per clip for each trait

def plotDistributionBox():
    data = meanScoresPerClip()[:,1:]
    df = pd.DataFrame(data,columns=bigFive) 
    plot2 = sns.catplot(kind='box', data=df,  aspect=2, palette=sns.color_palette("Set1",n_colors=5))
    plot2.set(title="Box plot of mean data per clip for each trait", xlabel="Big Five", ylabel = "Personality Perception Scores")
    

In [9]:
# HISTOGRAMS FOR EACH TRAIT FOR DISTRIBUTION OF MEAN SCORES PER CLIP
def distributionHistograms():
    data = meanScoresPerClip()[:,1:]
    df = pd.DataFrame(data,columns=bigFive) 
    display(Markdown('## Distribution of the mean scores per clip for every trait'))
    for i in range(5):
        plot = sns.displot(data[:,i], kde=True)
        plot.set_axis_labels(bigFive[i], "Count")

## Spearman Rank & Spearman Brown 

Agreement Between judges for a trait using Spearman Rank

In [10]:
#Spearman Rank Correlation Between judges

def SpearmanRankAllJudgesOneTraitRho(trait, gender):
    sexId = metadata.loc[:,['Clip_ID', 'Gender']].copy()
    
    listOfCoefficients = []
    listOfPvalues = []
    for i in range(11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        judgeA = pd.DataFrame(printJudge(x))
        withGenderA = pd.merge(judgeA, sexId, on='Clip_ID')
        for j in range(i+1,11):
            if (j<9):
                y = '0' + str(j+1)
            else:
                y = str(j+1)
            judgeB =  pd.DataFrame(printJudge(y))
            withGenderB = pd.merge(judgeB, sexId, on='Clip_ID')
            if(gender != 'both'):
                judgeA = withGenderA[withGenderA['Gender'] == gender]
                judgeB = withGenderB[withGenderA['Gender'] == gender]
            rho, p = scipy.stats.spearmanr(judgeA[trait].tolist(), judgeB[trait].tolist())
            listOfCoefficients.append(rho)
            listOfPvalues.append(p)
    
    return (listOfCoefficients)


Returns a list of [JudgeA, JudgeB, and then one by one the correlation coef & p value for each trait]

In [11]:
# no10 = True in order to exclude judge 10
def SpearmanRankAllJudgesAllTraits():
    listOfPairsAndRank = []
    i=0
    while (i<11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        judgeA = np.array(printJudge(x))
        j = i+1
        while (j<11):
            row =[]
            if (j<9):
                y = '0' + str(j+1)
            else:
                y = str(j+1)
            judgeB = np.array(printJudge(y))
            row.append(i+1)
            row.append(j+1)
            for k in range(1,6):
                row.append(np.array(scipy.stats.spearmanr(judgeA[:,k].tolist(), judgeB[:,k].tolist())))
            listOfPairsAndRank.append(row)
            j = j+1
        i = i+1
    return listOfPairsAndRank


TABLE OF ALL SPEARMAN RANK CORRELATIONS AND CORRESPONDING P VALUES FOR EACH TRAIT FOR ALL 55 UNIQUE COMBINATIONS OF JUDGES

In [12]:
#Returns a list [JudgeA, JudgeB, and then one by one the correlation coef OR p value for each trait]
def SpearmanRankAllJudgesAllTraitsRorP(RhoORP):
    if (RhoORP=='rho'):
        RorP = 0
    if (RhoORP=='p'):
        RorP = 1
    listOfPairsAndRank = []
    i=0
    while (i<11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        judgeA = np.array(printJudge(x))
        j = i + 1
        while (j<11):
            row =[]
            if (j<9):
                y = '0' + str(j+1)
            else:
                y = str(j+1)
            judgeB = np.array(printJudge(y))
            row.append(i+1)
            row.append(j+1)
            for k in range(1,6):
                row.append(scipy.stats.spearmanr(judgeA[:,k].tolist(), judgeB[:,k].tolist())[RorP])
            listOfPairsAndRank.append(row)
            j = j+1
        i = i+1
    return listOfPairsAndRank

def judgeSkewPlot():
    data = SpearmanRankAllJudgesAllTraitsRorP('p')

    display(Markdown('## Plots of P values (color)per combination of judges (axis)'))

    for i in range(1,6):
        d =  np.array(data)[:,i+1]
        pairs = list(zip(np.array(data)[:,0],np.array(data)[:,1]))
        stringPairs = list(map(lambda x: f'({x[0]}, {x[1]})', pairs))
        y1 = np.array(data)[:,0]
        y2 = np.array(data)[:,1]

        plt.scatter(np.array(data)[:,0], np.array(data)[:,1], c=d, cmap='viridis', s=100, alpha=1)
        cbar = plt.colorbar()
        cbar.set_label('P value')
        plt.xticks(np.arange(1,12))
        plt.yticks(np.arange(2,12))
        plt.title(bigFive[i-1])
        plt.xlabel('Judge A')
        plt.ylabel('Judge B')
        plt.show()


## Spearman Brown Correlation

In [13]:
# Getting the spearman brown for a specific trait for a specific number of judges
#Using the mean of rho values for each trait
def spearmanBrown(trait, numberOfjudges, gender):
    old = np.mean(SpearmanRankAllJudgesOneTraitRho(trait, gender))
    return (old, (numberOfjudges * old / (1 + (numberOfjudges - 1) * old)))

def SpearmanBrownTable(gender):
    number = 11 
    spearmanBrownPerTrait = []
    for trait in bigFive:       
        spearmanBrownPerTrait.append(spearmanBrown(trait,number,gender))
    dfBrown = pd.DataFrame(spearmanBrownPerTrait, columns = ['Correlation', 'Spearman Brown'])
    dfBrown.index = bigFive
    display(Markdown('#### Spearman Brown per Trait (Correlation (Reliability old) & Spearman Brown (new))'))
    display(dfBrown)
    return(dfBrown)


def SpearmanBrownResults(gender):
    data = SpearmanBrownTable(gender)
    values = data['Spearman Brown']
    rho = data['Correlation']
    for trait in bigFive:
        if (values[trait] > 0.7):
            print(trait, ' is GOOD \n     rho:', rho[trait])
        if (values[trait] > 0.5):
            print(trait, ' is FAIR \n     rho:', rho[trait])
        else:
            print(trait, ' is BAD \n     rho:', rho[trait])



In [14]:
#ALL RECORDINGS FROM ALL JUDGES (INCLUDING DUPLICATES) FOR A GENDER
def allJudgesOneGender(gender):
    total = splitPerGender(gender, judge1)
    i=1
    while (i<11):
        if (i<9):
            x = '0' + str(i+1)
        else:
            x = str(i+1)
        total = np.concatenate((total, splitPerGender(gender,np.array(printJudge(x)))), axis=0)
        i=i+1
    return total
    

## T test  

In [15]:
#TABLE OF T TEST VALUES FOR WHOLE DATA OF MALE VS FEMALE INCLUDING DUPLICATE CLIPS FROM ALL JUDGES

def tTestAllMalevsAllFemaleCorrection(method):
    data = tTestAllMalevsAllFemale()['P-value']
    if (method=='bonferroni'):
        alphaB= (statsmodels.stats.multitest.multipletests(data, alpha=0.05, method=method, maxiter=1, is_sorted=False, returnsorted=False)[3])
        return alphaB
    if (method=='sidak'):
        alphaS= (statsmodels.stats.multitest.multipletests(data, alpha=0.05, method=method, maxiter=1, is_sorted=False, returnsorted=False)[2])
        return alphaS

def tTestAllMalevsAllFemale():
    data=[]
    for j in range(1,6):
        data.append(scipy.stats.ttest_ind(a=allJudgesOneGender('F')[:,j].tolist(), b=allJudgesOneGender('M')[:,j].tolist(), equal_var=True))
    df = pd.DataFrame(data,columns=['T-statistic','P-value'])
    df.index = bigFive
    return df



In [16]:
def tTestAllDataResults(p):
    data = tTestAllMalevsAllFemale()
    display(Markdown('#### Statistical Significance for t-test over all male vs all female'))
    for i in range (5):
        value = data['P-value'][bigFive[i]]
        sig = ''
        if (value > p):
            sig = 'NOT'
        print(bigFive[i], " - ", sig, " Statistically Significant")


TABLE OF T TEST VALUES FOR AVERAGE CLIPS OF MALE VS FEMALE 
* AVERAGE PER CLIP OVER ALL JUDGES
* Judge independent

In [17]:
#TABLE OF T TEST VALUES FOR MEAN PER CLIP OF MALE VS FEMALE

def tTestAveragePerClipGender(averageType):
    if (averageType=="Mean"):
        averageClips = meanScoresPerClip()
    if (averageType=="Mode"):
        averageClips = modeScoresPerClip()
    if (averageType=="Median"):
        averageClips = medianScoresPerClip()
    data=[]
    for j in range(1,6):
        female = averageClips[np.where(np.array(metadata)[:,2] == 'F')[0]]
        male = averageClips[np.where(np.array(metadata)[:,2] == 'M')[0]]
        data.append(scipy.stats.ttest_ind(a=female[:,j].tolist(), b=male[:,j].tolist(), equal_var=True))

    df = pd.DataFrame(data,columns=['T-statistic','P-value'])
    df.index = bigFive
    return df


def tTestAveragePerClipGenderCorrection(method):
    data = tTestAveragePerClipGender('Mean')['P-value']
    if (method=='bonferroni'):
        alphaB= (statsmodels.stats.multitest.multipletests(data, alpha=0.05, method=method, maxiter=1, is_sorted=False, returnsorted=False)[3])
        return alphaB
    if (method=='sidak'):
        alphaS= (statsmodels.stats.multitest.multipletests(data, alpha=0.05, method=method, maxiter=1, is_sorted=False, returnsorted=False)[2])
        return alphaS
    
def tTestAveragePerClipGenderResults(averageType,p):
    data = tTestAveragePerClipGender(averageType)
    display(Markdown(f'#### Statistical Significance for t-test of  {averageType} average per clip'))
    for i in range (5):
        value = data['P-value'][bigFive[i]]
        sig = ''
        if (value > p):
            sig = 'NOT'
        print(bigFive[i], " - ", sig, " Statistically Significant")


In [18]:
def boxPlotAveragePerClipGender(averageType,alpha):
    display(Markdown('## Plot with average per clip for female vs male per trait'))
    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)

    for i in range(1, 6):
        
        if averageType == 'median':
            dfF = np.array(medianScoresPerClipGender('F')[:, i].tolist())
            dfM = np.array(medianScoresPerClipGender('M')[:, i].tolist())
        elif averageType == 'mean':
            dfF = np.array(meanScoresPerClipGender('F')[:, i].tolist())
            dfM = np.array(meanScoresPerClipGender('M')[:, i].tolist())

        colorsdfF = ['lightyellow'] * len(dfF)  
        colorsdfM = ['lightblue'] * len(dfM)  

        combinedData = np.concatenate((dfF, dfM), axis=0)
        labels = ['Female'] * len(dfF) + ['Male'] * len(dfM)

        df = pd.DataFrame({'Scores': combinedData, 'Gender': labels})

        sns.boxplot(x='Gender', y='Scores', data=df, ax=axes[i-1], hue='Gender', palette=[colorsdfF[0], colorsdfM[0]])
        sns.despine(offset = 5)

        # add statistical significance stars
        t, p = scipy.stats.ttest_ind(dfF, dfM)
        if (p<alpha):
            star = '*'
        else:
            star =''

        max =  np.max(combinedData)
        axes[i-1].text(0.5, max, star, ha='center', va='bottom')
        axes[i-1].set_title(f'{bigFive[i-1]}')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right')
    plt.suptitle("Average trait scores per recording - female vs male speakers",  fontsize=16)
    plt.tight_layout()
    plt.show()
